When deeper networks are able to start converging, a degradation problem has been exposed: with the network depth increasing, accuracy gets saturated (which might be unsurprising) and then degrades rapidly. Unexpectedly,such degradation is not caused by overﬁtting, and adding more layers to a suitably deep model leads to higher training error. <br>

Traditional Approach:
Standard networks aim to learn the desired underlying mapping H(x) directly through stacked layers.
Example: H(x)= desired output of layer or multiple layers for input x.

Residual Reformulation:
Instead of learning H(x), the network learns the residual function F(x)=H(x)−x. Here, F(x) is simply a function used to capture the difference between H(x) and x. 
The original mapping becomes H(x)=F(x)+x.

    Why?
    If the optimal H(x) is close to the identity mapping (i.e. H(x)≈x), it’s easier to optimize F(x)→0 than to fit H(x)=x through multiple nonlinear layers. This makes sense since in deeper layers, many layers might perform approximate identity mappings. Hence, their residuals with original input are almost 0 (i.e F(x) is almost 0).

Such a transformation is visualised by skip connections b/w layers which skip the input to end of layers and add the input there. This causes layers to approximate to 0, which is faster than letting the layers learn the actual underlying mapping.

IN THE PAPER, /2 REPRESENTS THAT THE RESULT IS DOWNSCALED BY 2. SO, A STRIDE OF 2 IS IMPLIED.


In [15]:
import numpy as np
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

In [16]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [17]:
import torchvision.datasets as datasets
mnist_trainset = datasets.MNIST(root=r'/home/xyphoes/Desktop/Projects/AIML - 2/Learning/Neural Networks/Projects/data', train=True, download=False, transform=None)
mnist_testset = datasets.MNIST(root=r'/home/xyphoes/Desktop/Projects/AIML - 2/Learning/Neural Networks/Projects/data', train=False, download=False, transform=None)

(xTrain, yTrain) = mnist_trainset.data.to(torch.float32), mnist_trainset.targets.to(torch.float32)
(xTest, yTest) = mnist_testset.data.to(torch.float32), mnist_testset.targets.to(torch.float32)
xTrain = xTrain / 255.0
xTest = xTest / 255.0
xTrain = xTrain.unsqueeze(1).to(device)  # shape: (N, 1, 28, 28)
xTest = xTest.unsqueeze(1).to(device)
yTest = yTest.long().to(device)
yTrain = yTrain.long().to(device)

In [18]:
class residualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride:int =1):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride
        self.CL1 = nn.Conv2d(in_channels,out_channels,stride=stride,kernel_size=3, padding=1)
        self.BN1 = nn.BatchNorm2d(out_channels)
        self.CL2 = nn.Conv2d(out_channels,out_channels,kernel_size=3, padding=1)
        self.BN2 = nn.BatchNorm2d(out_channels)
        self.pad_layer = None
        if(in_channels != out_channels or stride != 1):
            #pad layer has stride since mismatch occurs due to downsampling. hence, input must also be downsampled accordingly.
            self.pad_layer = nn.Conv2d(in_channels,out_channels, kernel_size=1, stride=stride)

    def forward(self, X):
        identity = X
        out = self.CL1(X)
        out = self.BN1(out)
        out = self.CL2(out)
        out = self.BN2(out)
        if(self.pad_layer is not None):
            identity = self.pad_layer(X)
        out = torch.add(out,identity)
        return out

In [19]:
class ResNET(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 7, 2, 3)           
        self.pool = nn.MaxPool2d(2, 2)                       
        self.layer1 = nn.ModuleList([residualBlock(64, 64) for _ in range(3)])
        self.trans1 = residualBlock(64,128,stride=2)
        self.layer2 = nn.ModuleList([residualBlock(128,128) for _ in range(3)])
        self.trans2 = residualBlock(128,256,stride=2)
        self.layer3 = nn.ModuleList([residualBlock(256,256) for _ in range(5)])
        self.fc = nn.Linear(256,10)
    def forward(self, X):
        out = self.pool(F.relu(self.conv1(X)))
        for block in self.layer1:
            out = F.relu(block(out))

        out = F.relu(self.trans1(out))
        for block in self.layer2:
            out = F.relu(block(out))

        out = F.relu(self.trans2(out))
        for block in self.layer3:
            out = F.relu(block(out))
        out = self.pool(out)
        out = out.view(out.size(0), -1) # flatten before passing to fc layer
        # !!!! Don't use relu on final FC layer since cross entropy expects logits only.
        # Using ReLU would make all -ve logits 0 and hence hurt training drastically. 
        out = self.fc(out)
        return out
    def predict_proba(self, X):
        logits = self.forward(X)
        probas = F.softmax(logits, 1)
        return probas

In [20]:
from torch.utils.data import TensorDataset, DataLoader
train_ds = TensorDataset(xTrain, yTrain)
batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

model = ResNET()
model = model.to(device)
optim = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()
n_epoch = 1

In [21]:
model.train()
for epoch in range(n_epoch):
    for xb,yb in train_loader:
        optim.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits,yb)
        loss.backward()
        optim.step()
    print(f"Error at epoch: {epoch+1} is {loss}")
model.eval()

KeyboardInterrupt: 